
# Deploying scGPT Foundation Models with MLflow – Extended Notebook Guide

This notebook is the extended, `code‑first` (?) companion to:

- Deploying Foundation Models with MLflow & Databricks Model Serving
- (Short blog – conceptual overview)


Here, we focus on the full implementation:

- Wrapping scGPT in a custom mlflow.pyfunc.PythonModel
- Embedding preprocessing and inference into predict()
- Designing a JSON‑serializable AnnData input format
- Logging to MLflow and registering to Unity Catalog
- Deploying to Databricks Model Serving
- Monitoring with AI Gateway inference tables
- Troubleshooting common issues

# 0. Prerequisites

You should already have:   

- A trained scGPT model (weights + config + vocab)  
- A Databricks workspace with:
    - Unity Catalog enabled 
    - Model Serving enabled
- A working short blog as conceptual reference 

We’ll assume:     
    
<br>

```
catalog = "your_catalog"
schema  = "your_schema"
model_name = "scgpt_gene_embeddings"
registered_model_name = f"{catalog}.{schema}.{model_name}"
```

# 1. Environment Setup  

<br>

```
# Install pinned versions (run once in a cluster init or %pip)
%pip install scgpt==0.2.4
%pip install flash-attn==2.5.9.post1
%pip install wandb==0.19.11
%pip install mlflow
```
<br>

```
import json
from typing import Dict


import mlflow
import mlflow.pyfunc
from mlflow.models import infer_signature
import mlflow.types


import numpy as np
import pandas as pd
import torch


import scanpy
import scipy
from anndata import AnnData

from scgpt.tokenizer.gene_tokenizer import GeneVocab
from scgpt.model import TransformerModel
from scgpt.preprocess import Preprocessor
```  
  

**Why pin versions?**

Databricks Model Serving builds a fresh environment from your conda_env. Pinning versions of torch, scgpt, and flash-attn ensures that:

- The notebook and serving environments match
- You avoid “works in notebook, breaks in serving” surprises

# 2. Preparing scGPT Artifacts

We assume you already have:

```
model_file_path   = "/dbfs/path/to/scgpt_weights.pt"
config_file_path  = "/dbfs/path/to/scgpt_config.json"
vocab_file_path   = "/dbfs/path/to/gene_vocab.json"
```   

These will be logged as MLflow artifacts.

**Why artifacts via MLflow?**

MLflow:
- Copies these files into the model’s artifact directory
- Makes them available (via context.artifacts) wherever the model is loaded:
    - Notebooks
    - Batch jobs
    - Model Serving
    - This makes the PyFunc model self‑contained and portable.


# 3. Implementing the Custom PyFunc Wrapper

We wrap scGPT as a PythonModel so MLflow and Model Serving can call predict() on it.   

## 3.1 Skeleton: TransformerModelWrapper  
<br>

```
import mlflow.pyfunc
import mlflow.types

class TransformerModelWrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, special_tokens=["<pad>", "<cls>", "<eoc>"]):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.special_tokens = special_tokens


    def load_context(self, context):
        # Artifact paths
        self.model_file = context.artifacts["model_file"]
        self.model_config_file = context.artifacts["model_config_file"]
        self.vocab_file = context.artifacts["vocab_file"]


        # Vocabulary
        self.vocab = GeneVocab.from_file(self.vocab_file)
        for s in self.special_tokens:
            if s not in self.vocab:
                self.vocab.append_token(s)
        self.gene2idx = self.vocab.get_stoi()
        self.ntokens = len(self.vocab)


        # Config
        with open(self.model_config_file, "r") as f:
            self.model_configs = json.load(f)


        self.embsize      = self.model_configs["embsize"]
        self.nhead        = self.model_configs["nheads"]
        self.d_hid        = self.model_configs["d_hid"]
        self.nlayers      = self.model_configs["nlayers"]
        self.n_layers_cls = self.model_configs["n_layers_cls"]
        self.pad_value    = self.model_configs["pad_value"]
        self.mask_value   = self.model_configs["mask_value"]
        self.n_bins       = self.model_configs["n_bins"]
        self.n_hvg        = self.model_configs["n_hvg"]
        
``` 

**Key design details:**

`__init__` is kept lightweight; no artifact loading there.
load_context is called by MLflow when the model is loaded, and gets a context with artifacts.
Vocabulary and config are loaded once per process, not per request.

We still need:
- `preprocess(...)`
- `filter(...)`
- `predict(...)`


# 4. Designing a Serving‑Compatible Input Format

scGPT expects AnnData with a sparse matrix and DataFrames (obs, var) – none of which are JSON‑serializable.   
Model Serving, however, expects JSON over HTTP.

## 4.1 Notebook vs. Serving formats
Notebook‑friendly (not JSON‑serializable):

<br>

```
adata_subset = adata[:100, :100]


input_data_dev = pd.DataFrame({
    "adata_sparsematrix": [adata_subset.X],   # csr_matrix
    "adata_obs": [adata_subset.obs],          # DataFrame
    "adata_var": [adata_subset.var],          # DataFrame
})
```

This is convenient for local experiments but cannot go over HTTP.
Serving‑friendly (JSON‑serializable):


<br>

```

adata_subset = adata[:100, :100]


input_data = pd.DataFrame({
    "adata_sparsematrix": [adata_subset.X.toarray().tolist()],         # list of lists
    "adata_obs": [adata_subset.obs.to_json(orient="split")],           # JSON string
    "adata_var": [adata_subset.var.to_json(orient="split")],           # JSON string
})
``` 

Notes:

- `csr_matrix → .toarray().tolist() → nested Python lists` 
- `DataFrame → .to_json(orient="split") → reversible JSON string`
- `Outer [ ... ]` in each column makes this a **single‑row** DataFrame, so `input_data["adata_sparsematrix"][0]` is the actual matrix representation.

This exact structure is what we’ll use for:
- input_example when logging with MLflow
- validate_serving_input
- Production HTTP payloads

# 5. Implementing `preprocess()` and `predict()`

## 5.1 Preprocessing: reconstruct AnnData + run scGPT Preprocessor
<br>   

``` 

class TransformerModelWrapper(mlflow.pyfunc.PythonModel):
    # __init__ and load_context as above ...


    def preprocess(
        self,
        context,
        input_data_path: str = None,
        input_dataframe: pd.DataFrame = None,
        data_is_raw: bool = False,
        params: dict | None = None,
    ) -> AnnData:
        if params is None:
            params = {}


        # Case 1: load AnnData from file (jobs/offline)
        if input_data_path and input_dataframe is None:
            loaded_data = scanpy.read(str(input_data_path), cache=True)
            ori_batch_col = "batch"
            loaded_data.obs["celltype"] = loaded_data.obs["final_annotation"].astype(str)


        # Case 2: reconstruct AnnData from serialized DataFrame (serving)
        elif input_dataframe is not None and isinstance(input_dataframe, pd.DataFrame):
            adata_sparsematrix = scipy.sparse.csr_matrix(
                input_dataframe["adata_sparsematrix"][0]
            )
            adata_obs = pd.read_json(input_dataframe["adata_obs"][0], orient="split")
            adata_var = pd.read_json(input_dataframe["adata_var"][0], orient="split")


            loaded_data = scanpy.AnnData(
                adata_sparsematrix, obs=adata_obs, var=adata_var
            )
            ori_batch_col = "batch"
            loaded_data.obs["celltype"] = loaded_data.obs["final_annotation"].astype(str)
        else:
            raise ValueError("No input data provided.")


        self.data_is_raw = params.get("data_is_raw", data_is_raw)


        preprocessor = Preprocessor(
            use_key=params.get("use_key", "X"),
            filter_gene_by_counts=params.get("filter_gene_by_counts", 3),
            filter_cell_by_counts=params.get("filter_cell_by_counts", False),
            normalize_total=params.get("normalize_total", 1e4),
            result_normed_key=params.get("result_normed_key", "X_normed"),
            log1p=params.get("log1p", self.data_is_raw),
            result_log1p_key=params.get("result_log1p_key", "X_log1p"),
            subset_hvg=params.get("subset_hvg", self.n_hvg),
            hvg_flavor=params.get(
                "hvg_flavor",
                "seurat_v3" if self.data_is_raw else "cell_ranger",
            ),
            binning=params.get("binning", self.n_bins),
            result_binned_key=params.get("result_binned_key", "X_binned"),
        )


        self.n_input_bins = params.get("binning", self.n_bins)
        preprocessor(loaded_data, batch_key=ori_batch_col)
        return loaded_data
   
```

## 5.2 Optional gene filtering

<br>

```   

def filter(
        self, gene_embeddings: np.ndarray, preprocessed_data: AnnData
    ) -> Dict[str, np.ndarray]:
        return {
            gene: gene_embeddings[i]
            for i, gene in enumerate(self.gene2idx.keys())
            if gene in preprocessed_data.var.index.tolist()
        }
   
```        

## 5.3 Predict: end‑to‑end pipeline

<br>

``` 
def predict(
        self,
        context,
        model_input: pd.DataFrame = None,
        params: Dict[str, mlflow.types.DataType] | None = None,
    ) -> Dict[str, np.ndarray]:
        if params is None:
            params = {}


        need_preprocess = params.get("need_preprocess", True)


        # 1) Optional preprocessing
        if need_preprocess:
            if model_input is None and params.get("input_data_path") is None:
                raise ValueError(
                    "Either 'model_input' or 'input_data_path' must be provided."
                )
            preprocessed_data = self.preprocess(
                context,
                input_data_path=params.get("input_data_path"),
                input_dataframe=model_input,
                data_is_raw=params.get("data_is_raw", False),
                params=params,
            )


        # 2) Build model
        self.model = TransformerModel(
            ntoken=params.get("ntokens", self.ntokens),
            d_model=params.get("embsize", self.embsize),
            nhead=params.get("nhead", self.nhead),
            d_hid=params.get("d_hid", self.d_hid),
            nlayers=params.get("nlayers", self.nlayers),
            vocab=self.vocab,
            pad_value=params.get("pad_value", self.pad_value),
            n_input_bins=params.get(
                "n_input_bins",
                getattr(self, "n_input_bins", self.n_bins),
            ),
        )


        # 3) Load weights robustly
        state = torch.load(self.model_file, map_location=self.device)
        try:
            self.model.load_state_dict(state)
        except RuntimeError:
            model_dict = self.model.state_dict()
            pretrained_dict = {
                k: v
                for k, v in state.items()
                if k in model_dict and v.shape == model_dict[k].shape
            }
            model_dict.update(pretrained_dict)
            self.model.load_state_dict(model_dict)


        self.model.to(self.device)
        self.model.eval()


        # 4) Encode all genes
        gene_ids = np.array(list(self.gene2idx.values()))
        with torch.no_grad():
            gene_embeddings = self.model.encoder(
                torch.tensor(gene_ids, dtype=torch.long).to(self.device)
            ).detach().cpu().numpy()


        # 5) Optionally filter by input genes
        if need_preprocess:
            gene_embeddings_dict = self.filter(gene_embeddings, preprocessed_data)
        else:
            gene_embeddings_dict = {
                gene: gene_embeddings[i]
                for i, gene in enumerate(self.gene2idx.keys())
            }

        return gene_embeddings_dict
   
```        
**Why embed preprocessing inside predict()?**

- Model Serving only calls predict().  
- If preprocessing is not reachable via predict(), serving cannot run it.
- This pattern keeps dev and prod behavior aligned:   
    - Same code path in notebooks and endpoints
    - Configured via params rather than different entry points

# 6. Parameters and Signature

## 6.1 Define a parameter set  

<br> 

``` 

params = {
    "need_preprocess": True,
    "input_data_path": None,
    "data_is_raw": False,
    "use_key": "X",
    "filter_gene_by_counts": 3,
    "filter_cell_by_counts": False,
    "normalize_total": 1e4,
    "result_normed_key": "X_normed",
    "log1p": False,
    "result_log1p_key": "X_log1p",
    "subset_hvg": 1200,
    "hvg_flavor": "cell_ranger",
    "binning": 51,
    "result_binned_key": "X_binned",
    "embsize": 512,
    "nhead": 8,
    "d_hid": 512,
    "nlayers": 12,
    "n_layers_cls": 3,
}

```

MLflow parameter rules:
- Must be JSON‑serializable: bool, int, float, str, lists/dicts of these
- Flat dict: no deeply nested structures
- Keys are strings


## 6.2 Create a serving‑compatible input example

<br>

``` 

adata_subset = adata[:100, :100]

input_data = pd.DataFrame({
    "adata_sparsematrix": [adata_subset.X.toarray().tolist()],
    "adata_obs": [adata_subset.obs.to_json(orient="split")],
    "adata_var": [adata_subset.var.to_json(orient="split")],
})
   
``` 

## 6.3 Infer the MLflow model signature

<br>

``` 

signature = infer_signature(
    model_input=input_data,
    model_output={"gene_embeddings": np.random.rand(100, 512)},  # example
    params=params,
)
   
```

**What does the signature buy you?**

Encodes the contract for:
- Input schema
- Output schema
- Parameter types

Powers:
- Validation (validate_serving_input)
- Documentation (UI shows input/output schemas)
- Early failure for mismatched inputs

# 7. Logging and Registering the Model (MLflow + UC)

```  

mlflow.set_registry_uri("databricks-uc")


with mlflow.start_run():
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=TransformerModelWrapper(
            special_tokens=["<pad>", "<cls>", "<eoc>"]
        ),
        artifacts={
            "model_file": str(model_file_path),
            "model_config_file": str(config_file_path),
            "vocab_file": str(vocab_file_path),
        },
        conda_env={
            "name": "mlflow-env",
            "channels": ["conda-forge"],
            "dependencies": [
                "python=3.11.11",
                "pip",
                {
                    "pip": [
                        "numpy==1.26.4",
                        "torch==2.0.1+cu118",
                        "torchvision==0.15.2+cu118",
                        "scgpt==0.2.4",
                        "flash-attn==2.5.8+cu118",
                        "wandb==0.19.11",
                    ],
                },
            ],
        },
        signature=signature,
        input_example=(input_data, params),  # note: tuple
        registered_model_name=registered_model_name,
    )
  
```     

This does two things in one step:
- Logs a run with artifacts, env, signature, and input example.
- Registers a new version under:   
    `catalog.schema.scgpt_gene_embeddings`

# 8. Local Testing and Serving Validation

## 8.1 Load and test in a notebook

<br> 

``` 

model_uri = f"models:/{registered_model_name}/1"  # version 1


model = mlflow.pyfunc.load_model(model_uri)


predictions = model.predict(
    data=input_data,
    params={
        "need_preprocess": True,
        "input_data_path": None,
        "data_is_raw": False,
        # you can override/extend other params here
    },
)
  
```  

## 8.2 Validate serving input

<br> 

```

serving_input_example = mlflow.models.convert_input_example_to_serving_input(
    (input_data, params)
)


mlflow.models.validate_serving_input(
    model_uri,
    serving_input_example,
)
  
```  

`convert_input_example_to_serving_input` encodes `(data, params)` into the exact JSON envelope Model Serving expects; `validate_serving_input` checks that against the model signature.
If both steps pass, you’re safe to deploy.


# 9. Deploying to Databricks Model Serving

## 9.1 Recommended: Databricks SDK for Python

<br>

```

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput


w = WorkspaceClient()


endpoint = w.serving_endpoints.create(
    name="scgpt-serving",
    config=EndpointCoreConfigInput(
        served_entities=[
            ServedEntityInput(
                entity_name=registered_model_name,  # catalog.schema.scgpt_gene_embeddings
                entity_version="1",
                workload_size="Medium",
                scale_to_zero_enabled=False,
                workload_type="GPU_SMALL",  # scGPT is transformer-based
            )
        ]
    ),
)

w.serving_endpoints.wait_get_serving_endpoint_not_updating(endpoint.name)
  
```

**Why SDK?**
- Type‑safe, IDE‑friendly
- Easy to use in CI/CD and jobs
- Declarative endpoint configuration in code

(Alternatives: MLflow Deployments SDK, UI, REST – see short blog for comparison.)


# 10. Making Inference Requests

We reuse the same `(input_data, params) `structure:

<br>

```   
  
import requests
import mlflow


adata_subset = adata[:50, :100]


input_data = pd.DataFrame({
    "adata_sparsematrix": [adata_subset.X.toarray().tolist()],
    "adata_obs": [adata_subset.obs.to_json(orient="split")],
    "adata_var": [adata_subset.var.to_json(orient="split")],
})


params = {
    "need_preprocess": True,
    "input_data_path": None,
    "data_is_raw": False,
    "use_key": "X",
    "filter_gene_by_counts": 3,
    "filter_cell_by_counts": False,
    "normalize_total": 1e4,
    "result_normed_key": "X_normed",
    "log1p": False,
    "result_log1p_key": "X_log1p",
    "subset_hvg": 1200,
    "hvg_flavor": "cell_ranger",
    "binning": 51,
    "result_binned_key": "X_binned",
}


serving_input = mlflow.models.convert_input_example_to_serving_input(
    (input_data, params)
)


response = requests.post(
    f"https://{databricks_host}/serving-endpoints/scgpt-serving/invocations",
    headers={
        "Authorization": f"Bearer {YOUR_DATABRICKS_TOKEN}",
        "Content-Type": "application/json",
    },
    data=serving_input,
)


if response.status_code != 200:
    raise RuntimeError(f"Request failed: {response.status_code} {response.text}")


predictions = response.json()
  
```

<br>

Typical response shape:

<br>

``` 
   
{
  "predictions": [
    {
      "gene_embeddings": [[0.123, 0.456, ...], ...]
    }
  ]
}
   
```   

You can then:
- Save embeddings to Delta
- Feed them into downstream models
- Visualize them in notebooks or BI tools

# 11. Monitoring with AI Gateway Inference Tables

For production, you need visibility into:
- Latency
- Error rates
- Request/response payloads
- Who is using the endpoint and which model versions

Databricks AI Gateway can auto‑capture this into inference tables.

## 11.1 Enable AI Gateway at creation time

```
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    AiGatewayConfig,
    AiGatewayInferenceTableConfig,
    AiGatewayUsageTrackingConfig,
)

w = WorkspaceClient()

endpoint = w.serving_endpoints.create(
    name="scgpt-serving",
    config=EndpointCoreConfigInput(
        name="scgpt-serving",
        served_entities=[
            ServedEntityInput(
                entity_name=registered_model_name,
                entity_version="1",
                workload_size="Medium",
                scale_to_zero_enabled=False,
                workload_type="GPU_SMALL",
            )
        ],
    ),
    ai_gateway=AiGatewayConfig(
        inference_table_config=AiGatewayInferenceTableConfig(
            catalog_name=catalog,
            schema_name=schema,
            table_name_prefix="scgpt_inference",
            enabled=True,
        ),
        usage_tracking_config=AiGatewayUsageTrackingConfig(enabled=True),
    ),
)
``` 
This will create a table like:
`{catalog}.{schema}.scgpt_inference_payload`
with columns such as:

- databricks_request_id
- request_date, request_time
- status_code
- execution_duration_ms
- request, response
- served_entity_id, requester

## 11.2 Query recent traffic

<br>

```

inference_df = spark.sql(f"""
  SELECT
    databricks_request_id,
    request_time,
    status_code,
    execution_duration_ms,
    requester,
    served_entity_id,
    request,
    response
  FROM {catalog}.{schema}.scgpt_inference_payload
  WHERE request_date >= current_date() - INTERVAL 7 DAYS
  ORDER BY request_time DESC
  LIMIT 100
""")


display(inference_df)
  
``` 

## 11.3 Performance metrics and error analysis

You can derive latency and error‑rate metrics directly from the AI Gateway inference table.  

<br>

```
     
performance_metrics = spark.sql(f"""
  SELECT 
    request_date,
    COUNT(*) AS total_requests,
    AVG(execution_duration_ms) AS avg_latency_ms,
    PERCENTILE(execution_duration_ms, 0.50) AS p50_latency_ms,
    PERCENTILE(execution_duration_ms, 0.95) AS p95_latency_ms,
    PERCENTILE(execution_duration_ms, 0.99) AS p99_latency_ms,
    SUM(CASE WHEN status_code != 200 THEN 1 ELSE 0 END) AS error_count,
    100.0 * SUM(CASE WHEN status_code != 200 THEN 1 ELSE 0 END) / COUNT(*) AS error_rate_pct,
    COUNT(DISTINCT served_entity_id) AS model_versions_used,
    COUNT(DISTINCT requester) AS unique_requesters
  FROM {catalog}.{schema}.scgpt_inference_payload
  WHERE request_date >= current_date() - INTERVAL 30 DAYS
  GROUP BY request_date
  ORDER BY request_date DESC
""")


display(performance_metrics)
  
```


Identify slow requests:

<br>

```
   
slow_requests = spark.sql(f"""
  SELECT 
    databricks_request_id,
    request_time,
    execution_duration_ms,
    status_code,
    served_entity_id,
    requester,
    request
  FROM {catalog}.{schema}.scgpt_inference_payload
  WHERE request_date >= current_date() - INTERVAL 1 DAYS
    AND execution_duration_ms > 5000  -- > 5 seconds
  ORDER BY execution_duration_ms DESC
  LIMIT 20
""")


display(slow_requests)
   
```   

Analyze error patterns:

<br>

``` 
   
error_analysis = spark.sql(f"""
  SELECT 
    status_code,
    COUNT(*) AS error_count,
    MIN(request_time) AS first_occurrence,
    MAX(request_time) AS last_occurrence,
    COUNT(DISTINCT requester) AS affected_users,
    COLLECT_LIST(databricks_request_id)[0:5] AS sample_request_ids,
    COLLECT_LIST(DISTINCT served_entity_id) AS affected_model_versions
  FROM {catalog}.{schema}.scgpt_inference_payload
  WHERE request_date >= current_date() - INTERVAL 7 DAYS
    AND status_code >= 400
  GROUP BY status_code
  ORDER BY error_count DESC
""")

display(error_analysis)
   
```

Recommended practice:   
- Build simple dashboards (e.g. in Databricks SQL) from these queries.
- Set alert thresholds for latency and error rates on production endpoints.

# 12. Common Pitfalls and How to Fix Them
This section is handy to keep at the bottom of the notebook for debugging.

## 12.1 Non‑serializable inputs

Symptom
- Errors like TypeError: Object of type csr_matrix is not JSON serializable
- 400/500 errors from the serving endpoint during request encoding
   
Cause
- Sending csr_matrix or DataFrame objects directly in the serving payload.

Fix
- Convert to JSON‑friendly types:

``` 
  
# WRONG
payload = {
    "adata_sparsematrix": adata.X,      # csr_matrix
    "adata_obs": adata.obs,             # DataFrame
    "adata_var": adata.var,             # DataFrame
}
  
``` 

<br>

```
# RIGHT
payload = {
    "adata_sparsematrix": adata.X.toarray().tolist(),   # list of lists
    "adata_obs": adata.obs.to_json(orient="split"),     # string
    "adata_var": adata.var.to_json(orient="split"),     # string
}

```

Wrap them in a one‑row DataFrame:

``` 

input_data = pd.DataFrame({k: [v] for k, v in payload.items()})

``` 
<br>

## 12.2 Wrong DataFrame shape / broadcasting

Symptom
- Indexing failures, weird shapes inside predict()
- Unexpected broadcasting or “lengths must match to broadcast” style errors

Cause
- Creating a DataFrame without outer list containers, e.g. pd.DataFrame({"col": value}) where value is a nested list.
- Fix

Always create a single‑row DataFrame:

<br>

```
   
# WRONG
pd.DataFrame({"adata_sparsematrix": adata.X.toarray().tolist()})
# creates many rows instead of 1


# RIGHT
pd.DataFrame({
    "adata_sparsematrix": [adata.X.toarray().tolist()],
    "adata_obs": [adata.obs.to_json(orient="split")],
    "adata_var": [adata.var.to_json(orient="split")],
})

```  


## 12.3 Invalid params (non‑JSON types)

Symptom
- MLflow or serving errors about parameters not matching the schema
- Serialization failures for params

Cause
- Using non‑JSON types in params (e.g. numpy arrays, custom objects, nested dicts that don’t match the signature).

Fix
- Keep params flat and JSON‑serializable (bool, int, float, str, lists/dicts of these).
- Align with what you used at infer_signature time.

Example:

<br>

```  

params = {
    "need_preprocess": True,
    "binning": 51,
    "subset_hvg": 1200,
    "embsize": 512,
    "nhead": 8,
    # ...
} 
 
``` 

## 12.4 Dependency mismatches

Symptom
- Model loads in notebook but fails in serving with import errors or shape mismatches.
- ModuleNotFoundError or crashes in load_context.

Cause
- Different versions of torch / scgpt / flash-attn between development and serving environments.

Fix
- Pin exact versions in conda_env in mlflow.pyfunc.log_model.
- Use the same versions in your cluster (%pip install ...) that you declare in conda_env.

## 12.5 Mixed logging configs (legacy vs AI Gateway)

Symptom
- Confusing or duplicate inference logs.
- Errors like “auto_capture_config and inference_table_config cannot both be set“.

Cause
- Mixing the legacy inference_table_config with the new auto_capture_config / AiGatewayConfig.

Fix
- For new endpoints, use only AI Gateway:

``` 
 
EndpointCoreConfigInput(
    served_entities=[...],
    auto_capture_config=AiGatewayConfig(
        inference_table_config=AiGatewayInferenceTableConfig(...),
        usage_tracking_config={"enabled": True},
    ),
)

```

## 12.6 Endpoint never becomes READY

Symptom
- Endpoint stuck in NOT_READY or CONFIG_UPDATE_PENDING for a long time.

Causes
- Large model taking time to load, or
- Errors while loading the model (e.g. missing dependencies).

Fix
- Use wait_get_serving_endpoint_not_updating with a timeout and inspect endpoint details:

``` 
  
from datetime import timedelta
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

w.serving_endpoints.wait_get_serving_endpoint_not_updating(
    "scgpt-serving",
    timeout=timedelta(minutes=30),
)
  
```

If it fails, fetch endpoint detail via SDK or REST and inspect error messages in the config/state fields.

## 12.7 GPU sizing issues / OOM

Symptom
- Out‑of‑memory errors (CUDA out of memory) or very slow inference.

Fix
- Start with:

```   

ServedEntityInput(
    entity_name=registered_model_name,
    entity_version="1",
    workload_type="GPU_SMALL",   # or GPU_MEDIUM / GPU_LARGE
    workload_size="Medium",
    scale_to_zero_enabled=False,
)
  
``` 
and then adjust based on:
- Latency and throughput requirements
- Size of the model and typical batch sizes


# 13. Why This Pattern Generalizes

This notebook/guide uses scGPT as the concrete model, but the pattern is reusable for other complex foundation models (e.g., Geneformer, scBERT, multimodal omics models):

Custom PyFunc wrapper:
- Encapsulates artifacts, config, and custom inference logic.
Preprocessing inside predict():
- Ensures Model Serving runs the full scientific pipeline.
Strict serialization:
- Convert scientific objects to JSON‑friendly primitives (lists, strings, scalars).
MLflow signature + input_example:
- Declare an explicit, validated serving contract.
MLflow + Unity Catalog:
- Central registry, governance, and reproducibility.
Databricks Model Serving + AI Gateway:
- Low‑latency GPU serving and built‑in observability.

Once you’re comfortable with this for scGPT, swapping in a different model usually means:
- Different artifacts (weights, vocab, configs)
- Different preprocessing steps
- Slightly different return structure from predict()

The MLflow / Databricks side stays largely the same.

# 14. References and Further Reading

MLflow
- Custom PyFunc Models
https://mlflow.org/docs/latest/models.html#custom-pyfunc-models
https://mlflow.org/docs/latest/python_api/mlflow.pyfunc.html
- Model Signatures & Input Examples
https://mlflow.org/docs/latest/models.html#model-signatures
https://mlflow.org/docs/latest/models.html#input-example
- MLflow Deployments
https://mlflow.org/docs/latest/python_api/mlflow.deployments.html

Databricks Model Serving / Mosaic AI
- Databricks Model Serving (classic + Mosaic AI)
https://docs.databricks.com/en/machine-learning/model-serving/index.html
- Custom Model Serving
https://docs.databricks.com/en/machine-learning/model-serving/custom-models.html
- Mosaic AI Model Serving (multi‑model, routing)
https://docs.databricks.com/en/generative-ai/model-serving/index.html
- Monitoring / Inference Tables / AI Gateway
https://docs.databricks.com/en/generative-ai/ai-gateway/index.html
https://docs.databricks.com/en/machine-learning/model-serving/inference-tables.html

scGPT and related models
- scGPT (Cui et al. 2023, bioRxiv)
https://github.com/bowang-lab/scGPT
- Geneformer
https://github.com/jaybee84/genformer
- scBERT
https://github.com/TencentAILabHealthcare/scBERT
